# Lab 5.05 - Two-sample t-test

In [ ]:
# Package imports
import numpy as np                                  # "Scientific computing"
import scipy.stats as stats                         # Statistical tests

import pandas as pd                                 # Dataframe
import matplotlib.pyplot as plt                     # Basic visualisation
from statsmodels.graphics.mosaicplot import mosaic  # Mosaic plot
import seaborn as sns                               # Advanced dataviz

## Exercise 5 - Comparing test results between groups

A large number of students participated in a test that was organized in several successive sessions. Because creating a separate test for each session was practically unfeasible, the same questions were used in each session. Consequently, there is a danger that students could pass on information about the test to the groups that still had to come after. Later groups then have an advantage over the first. However, this also evident from the test scores?

The file `test-results.csv` contains all results of the test. The score is a floating point number with maximum 40. If the score field is empty, this indicates a student that was absent. Each session is identified by a letter, in the order of the consecutive sessions.

- Day 1: sessions A, B
- Day 2: sessions C, D, E
- Day 3: sessions F, G, H

Sessions A and B were conducted on a different campus, so it could be assumed that there is little to no communication with students from other sessions.

If information was passed on successfully, we expect the scores of later groups to be significantly better than the first.

**Note** that the reverse reasoning does not necessarily hold: if it turns out that the result of later sessions is indeed significantly better, this does not necessarily mean that the (only) cause is passing on of information. There may also be other causes (e.g. “weaker” class groups happen to be scheduled earlier).

1. Explore the data. Calculate the appropriate measures for central tendency and dispersion for the full dataset and for each session individually.
2. Plot a bar graph of the average score per session with error bars denoting one standard deviation.
3. Make a box plot of the scores divided per group. Compare the sessions listed below. Do you think there is a significant difference between the results? Can we suspect that information has been passed on?
    - A and B
    - C, D and E
    - F, G and H
    - C and H
    - A and H
4. Use an appropriate statistical test to determine whether the differences between the groups listed above are also *significant*. Can we conclude that the later groups score better or not?

### Answers

The average score in each session was:

| Session | Average score |
| :-----: | :------------ |
|    A    | 13.1          |
|    B    | 17.2          |
|    C    | 18.8          |
|    D    | 22.5          |
|    E    | 18.9          |
|    F    | 17.8          |
|    G    | 18.7          |
|    H    | 20.9          |

The table below shows the p-value of a one-sided t-test of two samples between the given sessions:

| Alt. hypothesis                       | p-value   |
| :------------------------------------ | :-------- |
| $\overline{x}_A - \overline{x}_B < 0$ | 0.05356   |
| $\overline{x}_C - \overline{x}_D < 0$ | 0.01343   |
| $\overline{x}_E - \overline{x}_D < 0$ | 0.02356   |
| $\overline{x}_F - \overline{x}_H < 0$ | 0.05767   |
| $\overline{x}_G - \overline{x}_H < 0$ | 0.1156    |
| $\overline{x}_C - \overline{x}_H < 0$ | 0.1463    |
| $\overline{x}_A - \overline{x}_H < 0$ | 0.0003289 |

Remarks:

- The difference between **session A and B**, although it seems quite large, is insignificant for $\alpha = 0.05$
- **Sessions C, D and E**:
  - Session D had the highest score. Sessions C and E had similar results, at least the average score was similar.
  - Session D has a significantly higher score than either sessions C and E. However, session E came _after_ D, so the cause is definitely not the passing of information.
- The differences between **sessions F, G and H** insignificant
- **Sessions C and H** are respectively the first and the last session on the same campus. So, if there is an opportunity to pass on information about the test, there's definitely enough time between these two sessions. However, the difference is not significant!
- The difference between **sessions A and H** are significant, but since they took place on different campuses, it is questionable that this difference is caused by passing on information.

In [ ]:
lab5.04
lege Velden verwijderen
linux = linux.dropna()
linux.head()


groeperen het het gemiddelde berekenen
linux.groupby('Session')['Score'].mean()
linux.groupby('Session')['Score'].agg(['mean', 'std'])

plotten van de gegevens
sns.boxplot(data=linux, x='Score', y='Session',hue='Session')



stap1
H0  geen significant verschil in de scores
H1 de scores van F zijn hoger dan die van A

Stap 2
Alpha = 0.05

Stap 3
Independent
t = -2.47421678603158
Stap4
P waarde bepalen
t,p =stats.ttest_ind(a=linux[linux.Session=='A']['Score'],
                     b=linux[linux.Session=='F']['Score'],
                     alternative='less', equal_var=False)

print(f't = {t}')
print(f'p = {p}')
p = 0.009162997959891738

p is veel Kleiner dan alpha
voldoende reden om h 0 te verwerpen
de score van F is significant groter dan die van groep A

Zijn er significante verschillen tussen alle groepen

1.	ik heb 8 groepen met scores, welke test heb ik nodig om te weten of er significante verschillen zijn tussen de scores van de 8 groepen
ANOVA
ik heb 8 groepen met scores, hoe weet ik of de data per groep normaal verdeeld is.
Sharpiro wi


ik heb 8 groepen met scores, hoe weet ik of de data per groep normaal verdeeld is

for session_name in linux['Session'].unique():
    scores = linux[linux['Session'] == session_name]['Score']
    shapiro_test_stat, shapiro_p_value = stats.shapiro(scores)
    print(f"Session {session_name}:\n  Shapiro-Wilk Test Statistic = {shapiro_test_stat:.4f}\n  P-value = {shapiro_p_value:.4f}")
    if shapiro_p_value > 0.05:
        print("  The data for this session appears to be normally distributed (p > 0.05).")
    else:
        print("  The data for this session does not appear to be normally distributed (p <= 0.05).")
    print("\n")

geef mij een test die kan controleren of de variantie (spreiding) van de scores ongeveer gelijk is in alle groepen

# Extract scores for each session into a list
session_scores = [linux[linux['Session'] == session]['Score'] for session in linux['Session'].unique()]

# Perform Levene's test
levene_stat, levene_p_value = stats.levene(*session_scores)

print(f"Levene's Test Statistic = {levene_stat:.4f}")
print(f"P-value = {levene_p_value:.4f}")

if levene_p_value > 0.05:
    print("\nBased on Levene's test, the variances across sessions appear to be equal (p > 0.05).")
else:
    print("\nBased on Levene's test, the variances across sessions are significantly different (p <= 0.05).")

ik heb 8 groepen met scores, de punter per groep zijn normaal verdeeld. de varianties zijn echter niet homogeen. Welke test heb ik nogdig om te weten of er significante verschillen zijn tussen de scores van de 8 groepen

# Extract scores for each session into separate arrays
session_scores_A = linux[linux['Session'] == 'A']['Score']
session_scores_B = linux[linux['Session'] == 'B']['Score']
session_scores_C = linux[linux['Session'] == 'C']['Score']
session_scores_D = linux[linux['Session'] == 'D']['Score']
session_scores_E = linux[linux['Session'] == 'E']['Score']
session_scores_F = linux[linux['Session'] == 'F']['Score']
session_scores_G = linux[linux['Session'] == 'G']['Score']
session_scores_H = linux[linux['Session'] == 'H']['Score']

# Perform Welch's ANOVA (using f_oneway with equal_var=False)
f_stat_welch, p_value_welch = stats.f_oneway(session_scores_A,
                                             session_scores_B,
                                             session_scores_C,
                                             session_scores_D,
                                             session_scores_E,
                                             session_scores_F,
                                             session_scores_G,
                                             session_scores_H,
                                             equal_var=False) # Important for Welch's ANOVA

print(f"Welch's ANOVA F-statistic = {f_stat_welch:.4f}")
print(f"Welch's ANOVA P-value = {p_value_welch:.4f}")

if p_value_welch < 0.05:
    print("\nBased on Welch's ANOVA, there is a statistically significant difference between the means of at least some of the groups (p < 0.05).")
else:
    print("\nBased on Welch's ANOVA, there is no statistically significant difference between the means of the groups (p >= 0.05).")


Welch's ANOVA F-statistic = 4.6045
Welch's ANOVA P-value = 0.0002
